# Codon level profiling analysis
This is the 3rd step of the actual data analysis, and is used to visualize similarity between samples and conditions. <br>
**Run preprocessing & RiboSeq_MetageneAnalysis.ipynb first!** (can be run simultaneously with RiboSeq_GeneProfilingAnalysis.ipynb)

Requires **raw** WIG files (for all Riboseq samples), genome fasta/gb, & mapping offset chosen during Metagene analysis. **User must specify condition information**. Note: codon analysis only uses RiboSeq samples, not RNAseq

Also outputs .csv datatable with RPKM and log2FC by codon (with 11mer sequence) for each sample/condition

Runtime ~40min (longer with RNAseq datasets, because it just treats them the same as riboSeq datasets)

Can be run simultaneously with RiboSeq_GeneProfilingAnalysis.ipynb


#### Version info

2025-08
- seq heatmap
- pairwise heatmap
- trimer volcano plot
  
V2 10/23/2024 

## Imports

In [ ]:
import os
import time
import numpy as np
import copy

from matplotlib import pyplot as plt
import pandas as pd

from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.stats

from importlib import reload

import RiboSeq_Analysis_fxns as analysis
import RiboSeq_Analysis_plots as plot

starttime = time.time()

## User inputs

In [ ]:
user_inputs = { 
    
    # list names of all files to be analyzed - must have existing WIG files 
    # includes both RiboSeq & RNAseq
    'filenames': [ 'Rb1_DMSO', 'Rb1_TZD-10',  'Rb2_DMSO', 'Rb2_TZD-10', 
                   'Rn1_DMSO', 'Rn1_10xTZD',  'Rn2_DMSO', 'Rn2_10xTZD',  ],

    # currently only used to get a count of how many genes pass filters for all samples in a given replicate
    # but all metagene/gene plots after filter step can be easily modified to give replicate-specific figures using the genes_per_repetition dict
    'repNums': {'Rb1': ['Rb1_DMSO', 'Rb1_TZD-10', ], 
                'Rb2': ['Rb2_DMSO', 'Rb2_TZD-10', ],
                'Rn1': ['Rn1_DMSO', 'Rn1_10xTZD', ],
                'Rn2': ['Rn2_DMSO', 'Rn2_10xTZD', ], }, #{},

    'rnaSeq_pairs': {'Rb1_DMSO': 'Rn1_DMSO', 'Rb1_TZD-10': 'Rn1_10xTZD', 
                     
                      'Rb2_DMSO': 'Rn2_DMSO', 'Rb2_TZD-10': 'Rn2_10xTZD', 
                      
                    
                    'DMSO': 'Rn_DMSO', 'TZD-10': 'Rn_TZD-10', } , # { ribo1: rna1 },

    # must have a condition labeled either WT or DMSO to use as comparison for log fold change
    'condition_info': { 'DMSO': ['Rb1_DMSO', 'Rb2_DMSO'], 'TZD-10': ['Rb1_TZD-10','Rb2_TZD-10'], 
                        
                       'Rn_DMSO': ['Rn1_DMSO', 'Rn2_DMSO'], 'Rn_TZD-10': ['Rn1_10xTZD','Rn2_10xTZD'],
                        } , 
    
    # for each condition in condition info, label with either 'RiboSeq' or 'RNAseq'
    'condition_type': { 'DMSO': 'RiboSeq', 'TZD-10': 'RiboSeq', 
                        
                       'Rn_DMSO': 'RNAseq', 'Rn_TZD-10': 'RNAseq',
                        },
    
    # main directory for analysis & containing preprocessing files; analysis folder will be generated here
    'main_dir': '/wynton/home/fujimori/jkleinman/JKDF05_Rb-Rn/', 
    
    # directory containing your WIG files from preprocessing analysis for all samples in filenames
    'wig_dir':  '/wynton/home/fujimori/jkleinman/JKDF05_Rb-Rn/preprocessing/Wig/',

    # path of the genome fasta file - must be same fasta file as was used for genome alignment in preprocessing step
    'genome_fasta': '/wynton/home/fujimori/jkleinman/RiboSeq_code/genome_files/CP009273.1/CP009273.1.fasta', 

    # path of the genbank file which matches the genome fasta file; features will be extracted from this
    'genome_genbank': '/wynton/home/fujimori/jkleinman/RiboSeq_code/genome_files/CP009273.1/CP009273.1.gb', 
    
    # arbitrary value; used during visualization but not included in total sequence depth or avg gene reads so won't be included in normalization factor
    'utr_length_to_include': 50,
    
    # choose if you're using the Wig pileup at 5'/3' end of read or at center of read
    # In my experience 3pr gives much better peaks for both start & stop codon - also favored for bacterial riboSeq (Mohammad, Green, & Buskirk 2019)
    'alignment_type': '3pr', #'5pr' 'avg' or '3pr'
    
    # used for assigning P site; mapping offset =0 uses raw wig data
    # adjust the value after running the analysis so start codon peak aligns with 0 and stop codon peak aligns with -6 (stop codon in A-site at -3)
    # Note: mapping offset is generally a negative value -- JKDF05 riboSeq places P-site with -15 offset for 3pr pileup on both metagene plots
    'mapping_offset': -15,
    
    # used to plot normalized reads across a given window for a gene of interest -- ('geneName', [min, max])
    # note that if you include UTRs they must be ≤ the value for utr_length_to_include 
    # providing an empty list will give the full CDS window for the gene (no UTR) -- ('geneName', [])
    # 'genes_of_interest': [] 
    'genes_of_interest': [ ] ,

    # how long your shortest CDS can be
    'minCDS': 18,

    # number of nt at either end of genes to exclude from codon analysis
    'exclude_end_nt': 9
    
            }

In [ ]:
if len(user_inputs['rnaSeq_pairs'].keys()) != 0:
    RNAseq = True
else:
    RNAseq = False

## Run the analysis

#### 1. Build the output directories

In [ ]:
# check for & build the analysis directories

analysis_outdir = user_inputs['main_dir'] + time.strftime("%Y%m%d") + '_analysis/'
offset_dir = analysis_outdir + '{}_offset_{}/'.format(user_inputs['alignment_type'], user_inputs['mapping_offset'])
profiling_dir = offset_dir  + 'codonProfiling_minCDS{}_excEnd{}/'.format(user_inputs['minCDS'], user_inputs['exclude_end_nt'])
plot_dir = profiling_dir + 'plots/'
stall_dir = profiling_dir + 'stallPlots/'
plogo_dir = profiling_dir + 'pLogo/'


if not os.path.exists(analysis_outdir):
    print('Recommend running RiboSeq_MetageneAnalysis.ipynb first to choose appropriate mapping offset')
    os.mkdir(analysis_outdir)
if not os.path.exists(offset_dir):
    os.mkdir(offset_dir)
if not os.path.exists(profiling_dir):
    os.mkdir(profiling_dir)
if not os.path.exists(plot_dir):
    os.mkdir(plot_dir)
if not os.path.exists(stall_dir):
    os.mkdir(stall_dir)
if not os.path.exists(plogo_dir):
    os.mkdir(plogo_dir)

#### 2. load genome and Wig files

> Same as for Metagene analysis
> - CDS features only & excluding pseudogenes
> - UTRs of length utr_length_to_include on either end, not going to be used for this analysis

In [ ]:
### Load genome 

genome_dict = analysis.parse_genome(user_inputs)

# sanity check that gene info dictionaries are the same size 
print('CDS feature count (excluding pseudogenes:', len(genome_dict['full_sequence_dict']))
print('gene info dictionaries are same size:', len(genome_dict['full_sequence_dict'].keys()) == len(genome_dict['strand_dict'].keys()) == len(genome_dict['location_dict'].keys()))

### Load WIGs 

sample_files = {}

for fname in user_inputs['filenames']:
    sample_files[fname] = (user_inputs['wig_dir'] + fname +'_%s_fwd_fromSam.wig' % user_inputs['alignment_type'], 
                           user_inputs['wig_dir'] + fname +'_%s_rev_fromSam.wig' % user_inputs['alignment_type'])

sample_names = sample_files.keys()

### feature_dict_meta = { sample_name: {gene1:[reads by position], gene2: [], ...}
feature_dict_meta = analysis.read_in_wig_addOffset(sample_files, genome_dict, user_inputs)


#### 3. Calculate sample sequencing depth for normalization

> Calculate sequencing depth (total readCounts across entire genome) for each sample
>   - Using **CDS portions only** (exclude UTRs for purpose of normalization to read depth)
>   - value should be somewhat smaller than total readcount from bowtie alignment due to this restriction
>
> *Any reads in overlapping gene regions will be double counted -- hopefully relatively minimal effect*

In [ ]:
# check total read numbers across CDS portions only -- this will be the value used for normalization
# should be somewhat smaller than total read counts from WIG conversion in your preprocessing logs

utr_length_to_include = user_inputs['utr_length_to_include']

total_read_dict = {}
for sample in sample_names:
    all_features = []
    for read_list in feature_dict_meta[sample].values():
        all_features.extend(read_list[utr_length_to_include:-1*utr_length_to_include]) #read counts only in coding region - exclude UTR
    print(sample, np.sum(all_features))
    total_read_dict[sample] = np.sum(all_features)


#### 4. Filter by gene expression/coverage

> using the same filter method as during metagene analysis, with minCDS defined in user_inputs <br>
> Exclude genes from codon analysis if:
> - \> 80% of positions along gene = 0
> - nucleotide read average < 0.5 across CDS of gene
> - CDS < minCDS (18nt, 100nt, ?)

In [ ]:
#filter

# save an unfiltered copy of feature_dict_meta just in case
feature_dict_meta_full = copy.deepcopy(feature_dict_meta)

# remove genes which have low expression/coverage (>80% 0 or < 0.5 read average across CDS of gene) or have < 100nt CDS 
feature_dict_meta = analysis.filter_genes(sample_names, feature_dict_meta, user_inputs, user_inputs['minCDS'])


#### 5. Create a reference of common genes

>  genes common to all samples within a condition <br>
>  genes common to all samples within a condition AND WT together

In [ ]:
genes_in_all_samples, genes_per_condition, genes_per_conditionPlusWT = analysis.common_genes(sample_names, feature_dict_meta, genome_dict, user_inputs)


#### 6. Calculate pause scores across total length of gene & filter

> filters:
> - ONLY include genes common to all samples in condition + WT
> - exclude terminal (3) aa (exclude_end_nt = 9) -- exclude_end_nt defined in user_inputs; rec 9-30nt
> - exclude codons with < 5 total reads
>
> $$ PauseScore = Total Codon Reads / Total Gene Reads $$
>
> codon IDs use base ONE numbering for easier manual reference


In [ ]:
### pauseScore_meta = { condition: {sample1: {codon_id_1: pauseScore, codon_id2: pauseScore, ...} ,
###                                 sample2: {} ,
###                                 WT1: {codon_id_1: pauseScore, codon_id2: pauseScore, ...} ,
###                                 WT2: { } }}
reload(analysis)                                
pauseScore_meta = {}
codon_cpm_meta = {}
codon_counts_meta= {}

WT = ['DMSO' if 'DMSO' in user_inputs['condition_info'] else 'WT'][0]
expConditions = [condition  for condition in user_inputs['condition_info'] if condition not in ['WT', 'DMSO']]

for condition in expConditions:
    samples_to_consider = user_inputs['condition_info'][condition] + user_inputs['condition_info'][WT]
    common_genes = genes_per_conditionPlusWT[condition]
    
    pauseScore_meta[condition], codon_cpm_meta[condition], codon_counts_meta[condition] = analysis.codon_pauseScore(samples_to_consider, \
                                                                                      common_genes, feature_dict_meta, genome_dict, total_read_dict, \
                                                                                      user_inputs, user_inputs['exclude_end_nt'])

pauseScore_meta_all, codon_cpm_meta_all, codon_counts_meta_all = analysis.codon_pauseScore(sample_names, genes_in_all_samples, feature_dict_meta, \
                                                                    genome_dict, total_read_dict, user_inputs, user_inputs['exclude_end_nt'])

#### 7. Create a reference of common codons
> exclude any codons which don't appear in all samples for a grouping (e.g. WT + condition)

In [ ]:
codons_per_conditionPlusWT = {}

WT = ['DMSO' if 'DMSO' in user_inputs['condition_info'] else 'WT'][0]
WTsamples = user_inputs['condition_info'][WT]

for condition in pauseScore_meta:
    codons_per_conditionPlusWT[condition] = []
    for codon_id in pauseScore_meta[condition][WTsamples[0]].keys():
        temp_list = []
        for sample in pauseScore_meta[condition].keys():
            if codon_id not in pauseScore_meta[condition][sample]:
                temp_list.append(sample)
        if len(temp_list) == 0:
            codons_per_conditionPlusWT[condition].append(codon_id)

print('\n##################################\n')

for condition in expConditions:
    print('There are {} codons that appear in all {} + {} samples'.format(len(codons_per_conditionPlusWT[condition]), WT, condition))

codons_in_all_samples = []

for codon_id in pauseScore_meta_all[WTsamples[0]].keys():
    temp_list = []
    for sample in pauseScore_meta_all.keys():
        if codon_id not in pauseScore_meta_all[sample]:
            temp_list.append(sample)
    if len(temp_list) == 0:
        codons_in_all_samples.append(codon_id)
        
print('\n##################################\n')

print('There are {} codons that appear in all samples'.format(len(codons_in_all_samples)))

print('\n##################################\n')

#### 8. Average pause scores across condition
>only including genes which fit constraints in all WT samples and all experimental samples for each condition
><br>only including codons which fit constraints in all WT samples and all experimental samples for each condition

In [ ]:
avg_pauseScore_meta = analysis.average_pauseScore(pauseScore_meta, codons_per_conditionPlusWT, user_inputs)
avg_pauseScore_meta_all = analysis.average_pauseScore_all(pauseScore_meta_all, codons_in_all_samples, user_inputs)

#### 9. Calculate log2FC in pause score compared to WT & sort
> log2FC > 0 indicates stall in experimental condition; log2FC < 0 indicates stall in WT condition

In [ ]:
log2FC_pauseScore = analysis.pauseScore_log2FC(avg_pauseScore_meta, user_inputs)

sorted_log2FC_pauseScore = {}
for condition in log2FC_pauseScore:
    sorted_log2FC_pauseScore[condition] = {k: v for k, v in sorted(log2FC_pauseScore[condition].items(), key=lambda x: x[1], reverse=True)}


log2FC_pauseScore_all = analysis.pauseScore_log2FC_all(avg_pauseScore_meta_all, user_inputs)

sorted_log2FC_pauseScore_all = {}
for condition in log2FC_pauseScore_all:
    sorted_log2FC_pauseScore_all[condition] = {k: v for k, v in sorted(log2FC_pauseScore_all[condition].items(), key=lambda x: x[1], reverse=True)}

#### 10. Get cutoffs/list of sensitive & resistant codon sites
> selecting top/bottom 5th percentile of stall sites to evaluate sensitive/resistant sites <br>
> arbitrary cutoff, will also pull top/bottom 1000 sites 

In [ ]:

sensitive = {}
resistant = {}
for condition in sorted_log2FC_pauseScore:
    sensitive[condition]= {}
    resistant[condition] = {}

    sensitive_cutoff = np.percentile(list(sorted_log2FC_pauseScore[condition].values()), 95)
    resistant_cutoff = np.percentile(list(sorted_log2FC_pauseScore[condition].values()), 5)
    
    for codon_id in sorted_log2FC_pauseScore[condition].keys():
        if sorted_log2FC_pauseScore[condition][codon_id] > sensitive_cutoff:
            sensitive[condition][codon_id] = sorted_log2FC_pauseScore[condition][codon_id]

        elif sorted_log2FC_pauseScore[condition][codon_id] < resistant_cutoff:
            resistant[condition][codon_id] = sorted_log2FC_pauseScore[condition][codon_id]

    print('\n',condition, '-- %s codons passing filters' % len(sorted_log2FC_pauseScore[condition]))
    print('\t95th percentile: %s (%s codons) \t 5th percentile: %s (%s codons)' %\
          (np.percentile(list(sorted_log2FC_pauseScore[condition].values()), 95), len(sensitive[condition]), \
           np.percentile(list(sorted_log2FC_pauseScore[condition].values()), 5), len(resistant[condition])))
    


#### 11. Waterfall plot of log2FC codon values
> sensitive & resistant cutoffs (using 5th & 95th percentiles) are marked

In [ ]:
plot.codon_waterfall(sorted_log2FC_pauseScore, sensitive, resistant, plot_dir, user_inputs)

#### 12. Plot histogram of log2FC codon values
> sensitive & resistant cutoffs (using 5th & 95th percentiles) are marked

In [ ]:
plot.codon_histFC(sorted_log2FC_pauseScore, plot_dir, user_inputs)

#### 13. Get amino acid sequences by codon
> Assumes mapping to P-site<br>
> Produces a list of peptide sequences in the same order as their entries appear in input dictionary<br>
> - sequence contains 11 aa, beginning at -9 and ending at the A-site codon after the peak
> - eg: codon peak at M in alphabet gives DEFGHIJKLMN
>
> This fxn outputs text files containing peptide sequences to subfolder pLogo; **upload to pLogo webtool for visualization** (https://plogo.uconn.edu/)
> <br>Use 'sen'/'res' files as foreground with 'all' files as background

In [ ]:
aaSeq_dict = {}

# pLOGO by percentile cutoff
for condition in sorted_log2FC_pauseScore:
    aaSeq_dict[condition] = {}
    aaSeq_dict[condition]['sensitive'] = analysis.get_peak_seq(sensitive[condition], plogo_dir+'sen%s.txt' % condition, genome_dict, user_inputs)
    aaSeq_dict[condition]['all']       = analysis.get_peak_seq(sorted_log2FC_pauseScore[condition], plogo_dir+'all%s.txt' % condition, genome_dict, user_inputs)
    aaSeq_dict[condition]['resistant'] = analysis.get_peak_seq(resistant[condition], plogo_dir+'res%s.txt' % condition, genome_dict, user_inputs)

# pLOGO by top/bottom 1000 sites
for condition in sorted_log2FC_pauseScore:
    sen1000 = aaSeq_dict[condition]['sensitive'][:1000]
    res1000=aaSeq_dict[condition]['sensitive'][-1000:]

    with open(plogo_dir + condition + '_sen1000.txt' , 'w') as f:
            f.write('\n'.join(sen1000))
    with open(plogo_dir + condition + '_res1000.txt' , 'w') as f:
            f.write('\n'.join(res1000))            
   

#### 14. Visualizing amino acid frequency by position
> PLogo by heatmap of log2 fold change in reads per million<br>
> plot: AA x Position in 11mer peptide

In [ ]:
# plot aa frequency by position in nascent chain
plot.seq_hmap(codon_cpm_meta, genome_dict, plogo_dir, user_inputs)

#### 15. Pairwise analysis
> frequency at which aa's appear together (by FC or log2FC in RPM)<br>
> plot by position in nascent chain

In [ ]:
for condition in codon_cpm_meta:
    plot.hmap_grid(condition, codon_cpm_meta, plot_dir,genome_dict, user_inputs, transform=None)
    plot.hmap_grid(condition, codon_cpm_meta, plot_dir, genome_dict, user_inputs, transform='np.log2')

#### 16. Volcano plots
> trimer/tetramer frequency & pvals

In [ ]:
plot.volcano_trimer(codon_counts_meta, plot_dir, genome_dict, user_inputs)
#plot.volcano_tetramer(codon_counts_meta, plot_dir, genome_dict, user_inputs)

#### 17. Report top 20 stall sites with peptide sequence
> this is for visual comparison; all stall site information will be output to csv in the next step

In [ ]:
for condition in sorted_log2FC_pauseScore:
    print('\n#### top 20 stalls -- %s' % condition)
    print('CodonID\t\t\taaSeq\t\tlog2FC vs WT')
    #print(sensitive[condition])
    for entry in zip(list(sensitive[condition].keys())[:20], aaSeq_dict[condition]['sensitive'][:20], list(sensitive[condition].values())[:20]):
        print('\t'.join([str(x) for x in entry]))
    

#### 18. Calculate & plot average codon frequencies by position (all codons)
> individual plots for positions -1, 0, +1
> <br>log2FC in pause score compared to WT condition

In [ ]:
codon_freq, aa_freq = analysis.get_codon_aa_freq(sorted_log2FC_pauseScore_all, genome_dict, user_inputs)
plot.codon_frequency(codon_freq, plot_dir, user_inputs)
plot.aa_frequency(aa_freq, plot_dir, user_inputs)

#### 19. Save output to CSV file

In [ ]:

analysis.write_pauseScoreCSV(sorted_log2FC_pauseScore, aaSeq_dict, avg_pauseScore_meta, codon_cpm_meta, 
                             pauseScore_meta, genome_dict, profiling_dir, user_inputs)


#### 20. Plot stall window for top stalling genes

In [ ]:
top_stalls = []
for condition in sorted_log2FC_pauseScore:
    i = 0
    for codon_id in sorted_log2FC_pauseScore[condition].keys():
        if i <5:
            i += 1
            codon_id = codon_id.split('_')
            geneName = '_'.join(codon_id[:-1])
            codonNum = int(codon_id[-1])
            nt_pos = (codonNum - 1) *3
    
            window = [nt_pos - 30, nt_pos + 6]
            if window[0] < 0:
                window[0] = 0

            if geneName in genes_in_all_samples:
    
                if (geneName, window) not in top_stalls:
                   top_stalls.append((geneName, window))

top_stalls

In [ ]:

for gene_window in top_stalls:
    gene=gene_window[0]
    window=gene_window[1]
    
    # get the full name of a gene of interest
    geneID = analysis.gene_lookup(gene, genome_dict)

    plot.short_CDS_window_TE(geneID, sample_names, total_read_dict, feature_dict_meta, genome_dict, stall_dir, user_inputs, window, RNAseq)
    plot.short_CDS_window_rpkm(geneID, sample_names, total_read_dict, feature_dict_meta, genome_dict, stall_dir, user_inputs, window)
       

#### 21. Close out & save analysis
> from here, plot the exported stall sequences using the pLogo webtool (https://plogo.uconn.edu/)<br>
> Also analyze top stalls via the output CSV files 

In [ ]:
endtime = time.time()
print('Codon profiling analysis complete on {} samples in {} hh:mm:ss - alignment type: {}, mapping_offset: {}'.format(len(sample_names), \
            time.strftime("%H:%M:%S", time.gmtime(endtime-starttime)), user_inputs['alignment_type'], user_inputs['mapping_offset']))


In [ ]:
# sleep to allow notebook time to autosave all outputs 
time.sleep(120)
# produce an html copy of the notebook ; this is your log of the run & contains all code + cell outputs + plots 
os.system("jupyter nbconvert --to html RiboSeq_CodonAnalysis.ipynb")
os.rename('RiboSeq_CodonAnalysis.html', 
          profiling_dir + 'RiboSeq_CodonAnalysis_notebookLog_{}Alignment_{}offset.html'.format(user_inputs['alignment_type'], user_inputs['mapping_offset']))



# **Next: pLogo of exported stall sequences and look into csv file for top stalls**